# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds an honest, non-leaky feature vector for **Lane 2: Refresh / Content Opportunity Scoring** and conducts an explicit leakage vulnerability audit.

> Skill loaded: `skills/hunting-leakage-and-validating/SKILL.md` & `skills/flyrank/flyrank-data/SKILL.md`

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Load starter dataset
data_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

# 2. Define proxy label (is_declining_label)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"Base rate (declining pages): {base_rate:.4f} ({df['is_declining_label'].sum():,} / {len(df):,})")

# 3. Create missingness flags (to prevent blind fillna injecting category signals)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

# 4. Fill numeric missing values safely
df["search_volume_filled"] = df["search_volume"].fillna(0)
df["competition_filled"] = df["competition"].fillna(0)
df["cpc_filled"] = df["cpc"].fillna(0)
df["word_count_filled"] = df["word_count"].fillna(df["word_count"].median())

# 5. Feature Engineering
df["stale_flag"] = (df["days_since_last_update"] >= 180).astype(int)
df["high_impression_flag"] = (df["impressions_90d"] >= 500).astype(int)
df["impressions_per_day"] = df["impressions_90d"] / (df["content_age_days"] + 1)

# 6. One-hot encode categorical context
content_type_dummies = pd.get_dummies(df["content_type"], prefix="type", drop_first=True)
intent_dummies = pd.get_dummies(df["main_intent"].fillna("unknown"), prefix="intent", drop_first=True)

# 7. Assemble honest feature matrix X and target y
honest_numeric_cols = [
    "content_age_days", "days_since_last_update", "stale_flag",
    "impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d",
    "engaged_sessions_90d", "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume_filled", "competition_filled", "cpc_filled",
    "word_count_filled", "has_keyword_data", "has_word_count",
    "high_impression_flag", "impressions_per_day"
]

X = pd.concat([df[honest_numeric_cols], content_type_dummies, intent_dummies], axis=1)
X = X.loc[:, ~X.columns.duplicated()]
y = df["is_declining_label"]
groups = df["client_id"]

print(f"Honest feature matrix X shape: {X.shape}")
print("First 5 feature names:", list(X.columns[:5]))

Loaded dataset: 30,000 rows x 44 columns
Base rate (declining pages): 0.5421 (16,262 / 30,000)
Honest feature matrix X shape: (30000, 29)
First 5 feature names: ['content_age_days', 'days_since_last_update', 'stale_flag', 'impressions_90d', 'clicks_90d']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Dictionary & Availability Audit

| Feature Name | Meaning | Missing Value Strategy | Available Before Decision? |
|---|---|---|---|
| `content_age_days` | Days since page creation | No missing values (all >= 90) | Yes (historical property) |
| `days_since_last_update` | Days since content was last modified | No missing values | Yes (historical property) |
| `stale_flag` | Binary indicator: `days_since_last_update >= 180` | Derived flag (no missing) | Yes |
| `impressions_90d` | Trailing 90-day search impressions from GSC | No missing values | Yes (observed history) |
| `clicks_90d` | Trailing 90-day clicks from GSC | No missing values | Yes (observed history) |
| `ctr` | Search Click-Through Rate (x100 percentage) | No missing values | Yes (observed history) |
| `avg_position` | Average Google search ranking position | 0 means no data; kept as-is | Yes (observed history) |
| `sessions_90d` | Trailing 90-day GA4 sessions | No missing values | Yes (observed history) |
| `search_volume_filled` | Monthly keyword search volume | Imputed with 0 + `has_keyword_data` flag | Yes |
| `has_keyword_data` | Binary indicator if keyword metadata exists | Binary flag | Yes |
| `word_count_filled` | Article word count | Imputed with median + `has_word_count` flag | Yes |
| `type_*` | One-hot encoded content type | Categorical dummies | Yes |
| `intent_*` | One-hot encoded search intent | Categorical dummies | Yes |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [2]:
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Precision@K helper
def precision_at_k(probs, y_true, k=50):
    top_k_idx = np.argsort(-probs)[:k]
    return y_true.iloc[top_k_idx].mean()

# 1. Honest Grouped Split (GroupKFold on client_id)
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))

X_train_h, X_test_h = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 2. Honest Model Training
clf_honest = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_honest.fit(X_train_h, y_train)
y_prob_honest = clf_honest.predict_proba(X_test_h)[:, 1]
p50_honest = precision_at_k(y_prob_honest, y_test, k=50)
acc_honest = accuracy_score(y_test, clf_honest.predict(X_test_h))

# 3. Leaky Model Training (deliberately inserting label-derived trend_pct)
X_leaky = X.copy()
X_leaky["leaky_trend_pct"] = df["trend_pct"]
X_train_l, X_test_l = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]

clf_leaky = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leaky.fit(X_train_l, y_train)
y_prob_leaky = clf_leaky.predict_proba(X_test_l)[:, 1]
p50_leaky = precision_at_k(y_prob_leaky, y_test, k=50)
acc_leaky = accuracy_score(y_test, clf_leaky.predict(X_test_l))

print("=== LEAKAGE HUNT EXPERIMENT RESULTS ===")
print(f"Dataset Base Rate (Declining Pages): {base_rate:.4f}")
print(f"Honest Model  -> Accuracy: {acc_honest:.4f} | Precision@50: {p50_honest:.4f}")
print(f"Leaky Model   -> Accuracy: {acc_leaky:.4f} | Precision@50: {p50_leaky:.4f}")
print("\nConfession test: Leaky model jumps to 1.0000 because trend_pct directly reveals the label!")

=== LEAKAGE HUNT EXPERIMENT RESULTS ===
Dataset Base Rate (Declining Pages): 0.5421
Honest Model  -> Accuracy: 0.5755 | Precision@50: 0.5400
Leaky Model   -> Accuracy: 0.9997 | Precision@50: 1.0000

Confession test: Leaky model jumps to 1.0000 because trend_pct directly reveals the label!


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Fields Rationale

1. **`trend_direction`**: **Target Leakage.** The target variable `is_declining_label` is computed directly as `(trend_direction == "down")`. Using this feature makes prediction trivial and circular.
2. **`trend_pct`**: **Target Leakage.** `trend_direction` is derived directly from `trend_pct`. Including it allows the model to perfectly reconstruct the label.
3. **`content_id`**: **Pseudonym / Identifier.** Unique identifier per row; provides zero generalizable signal and causes memorization.
4. **`client_id`**: **Group Identifier.** Reserved strictly for GroupKFold client-holdout splits to test model generalization across unseen clients.
5. **`provider_used` & `model_used`**: **System Metadata.** Internal administrative/LLM generation tracking that does not reflect organic search performance or content decay signals.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.